# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library. We focus on high-level, code-driven data exploration using Croissant-structured datasets. All dataset elements are referenced by their `@id`.

### Dataset Source
Accessed via Croissant schema:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Fetch metadata (as object for display)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Let's review available `RecordSet` definitions, field definitions, and core Croissant `@id`s used throughout the schema.

**Note:** We will reference record sets, fields, and columns by their `@id` to ensure consistent and schema-driven processing. The Croissant library exposes schema objects via the `.metadata` properties, e.g. `dataset.metadata.record_sets`.

In [ ]:
# List record sets and their fields by @id
record_sets = dataset.metadata.record_sets
print("Record sets found (with @id):\n")
for rs in record_sets:
    print(f"- {rs.id} (type: {getattr(rs, 'type', None)})")
    print(f"  Label: {getattr(rs, 'label', None)}")
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields in this RecordSet (by @id):")
        for field in rs.fields:
            print(f"    - {field.id} ({field.name if hasattr(field, 'name') else ''})")
    print()

## 3. Data Extraction
Let's extract the tabular records from each main record set. We convert all records to DataFrames using the Croissant RecordSet's `@id`.

Select a record set from the above overview (by `@id`). If the dataset contains more than one, review each as needed.

In [ ]:
# Build a dictionary mapping record set @id to DataFrames
dataframes = {}
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

print(f"Found record sets: {record_set_ids}\n")

for record_set_id in record_set_ids:
    # Load all records for this record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Record set: {record_set_id}, {len(df)} records, columns: {df.columns.tolist()}")

# For illustration, let's pick the first record set for further analysis:
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"\nUsing main record set: {main_record_set_id}")
    df_main = dataframes[main_record_set_id]
    print("\nSample records from main record set:")
    display(df_main.head())

## 4. Exploratory Data Analysis (EDA)
Let's analyze and process a numeric field, filter records, and group by a categorical variable. We'll use only `@id` for selecting fields.

**Note:** Replace `<numeric_field_id>` and `<group_field_id>` below with the corresponding field IDs from the schema overview.

In [ ]:
# ---- User customization: set these IDs for your dataset structure ------

# You should set these from your schema overview (see previous cell output)
numeric_field_id = None
group_field_id = None

if main_record_set_id and not numeric_field_id:
    print("Please set `numeric_field_id` and `group_field_id` (if available) to the field @id from the schema. Example: 'gad7_total', 'village', etc.")
else:
    df = dataframes[main_record_set_id]
    # Ensure numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    # Remove outliers/out of range
    threshold = 10  # User can adjust
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by another field (if categorical field ID is provided)
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Let's look at the distribution of our numeric field and a group comparison if a grouping field is available.

In [ ]:
if main_record_set_id and numeric_field_id and numeric_field_id in df_main.columns:
    plt.figure(figsize=(7, 4))
    df_main[numeric_field_id].hist(bins=15, edgecolor='k', alpha=0.7)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id and group_field_id in df_main.columns:
        plt.figure(figsize=(10, 5))
        df_main[[group_field_id, numeric_field_id]].groupby(group_field_id).mean().plot(kind='bar')
        plt.title(f"Average {numeric_field_id} by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load and explore the [A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json) using the `mlcroissant` package. All dataset exploration referenced Croissant entity `@id` fields for robust, schema-aligned data processing.

- **Data Loading:** We loaded the dataset and inspected its record set/field structure programmatically.
- **Exploration:** Provided foundations for customized exploratory analysis and filtering.
- **Visualization:** Outlined approaches for distribution and group-wise comparisons.

For actual usage: set `numeric_field_id` (such as `'gad7_total'`, `'phq9_total'`, or corresponding field `@id`s for your mental health scores) and `group_field_id` (e.g., `'village'`, `'gender'`, or similar categorical field) to match the schema IDs shown in *Data Overview*.